In [ ]:
import numpy as np
import os
import glob
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from scipy.optimize import minimize

def piecewise_quadratic_fit_monotonic(x, y, threshold, mode='heating'):
    if mode == 'heating':
        mask = x < threshold
    else:
        mask = x > threshold
    x_fit = x[mask]
    y_fit = y[mask]
    if len(x_fit) < 5:
        return None, None, None

    def model_fn(params, x): return params[0]*x**2 + params[1]*x + params[2]
    def loss_fn(params): return np.sum((model_fn(params, x_fit) - y_fit)**2)

    X = np.vstack([x_fit**2, x_fit, np.ones_like(x_fit)]).T
    beta0 = np.linalg.lstsq(X, y_fit, rcond=None)[0]

    constraints = [
        {'type': 'eq', 'fun': lambda p: p[1] + 2*p[0]*threshold},
        {'type': 'eq', 'fun': lambda p: p[0]*threshold**2 + p[1]*threshold + p[2]},
        {'type': 'ineq', 'fun': lambda p: p[0]}
    ]

    result = minimize(loss_fn, beta0, constraints=constraints)
    if not result.success:
        return None, None, None

    params = result.x
    residuals = y_fit - model_fn(params, x_fit)
    mse = np.mean(residuals**2)
    cov = np.linalg.pinv(X.T @ X)
    param_se = np.sqrt(np.diag(mse * cov))

    ss_tot = np.sum((y_fit - np.mean(y_fit))**2)
    ss_res = np.sum(residuals**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return params, param_se, r2

def robust_fit(x, y, threshold, mode, county_code, hour, scale_factor=10.0):
    for scale_factor in [1.0, 10.0, 100.0, 1000.0]:
        result = piecewise_quadratic_fit_monotonic(x, y * scale_factor, threshold, mode)
        if result[0] is not None:
            params, ci, r2 = result
            return params / scale_factor, ci / scale_factor, r2
    return None, None, f"[fail] {mode}: {county_code}, hour={hour}"

def process_one_file(task):
    load_path, weather_path, save_path = task
    county_code = os.path.splitext(os.path.basename(load_path))[0].split('_')[-1]

    try:
        if os.path.exists(save_path):
            return None

        load_all = np.load(load_path)
        weather_all = np.load(weather_path)
        weather_data = weather_all['data']
        weather_aligned = weather_data[0:0 + 8760]
        temperature = weather_aligned[:, 5]

        heating_load = load_all['com_heating'] / 1000
        cooling_load = load_all['com_cooling'] / 1000

        heat_by_hour = heating_load.reshape((365, 24))
        cool_by_hour = cooling_load.reshape((365, 24))
        temp_by_hour = temperature.reshape((365, 24))

        heating_stats = np.full((24, 7), np.nan)
        cooling_stats = np.full((24, 7), np.nan)
        heating_thresholds = np.full((24,), np.nan)
        cooling_thresholds = np.full((24,), np.nan)

        for hour in range(24):
            x = temp_by_hour[:, hour]
            y_heat = heat_by_hour[:, hour]
            y_cool = cool_by_hour[:, hour]

            ph150, ci_h150, r2h150_or_msg = robust_fit(x, y_heat, threshold=150, mode='heating', county_code=county_code, hour=hour)
            ph300, ci_h300, r2h300_or_msg = robust_fit(x, y_heat, threshold=300, mode='heating', county_code=county_code, hour=hour)

            if ph150 is not None and (ph300 is None or r2h150_or_msg >= r2h300_or_msg):
                ph_final, ci_h_final, r2h_final = ph150, ci_h150, r2h150_or_msg
                heating_thresholds[hour] = 150
            elif ph300 is not None:
                ph_final, ci_h_final, r2h_final = ph300, ci_h300, r2h300_or_msg
                heating_thresholds[hour] = 300
            else:
                return f"[fail] heating two threshold failed: {county_code}, hour={hour}"

            heating_stats[hour, :] = np.concatenate([ph_final, ci_h_final, [r2h_final]])

            pc0, ci_c0, r2c0_or_msg = robust_fit(x, y_cool, threshold=0, mode='cooling', county_code=county_code, hour=hour)
            pc75, ci_c75, r2c75_or_msg = robust_fit(x, y_cool, threshold=75, mode='cooling', county_code=county_code, hour=hour)

            if pc0 is not None and (pc75 is None or r2c0_or_msg >= r2c75_or_msg):
                pc_final, ci_c_final, r2c_final = pc0, ci_c0, r2c0_or_msg
                cooling_thresholds[hour] = 0
            elif pc75 is not None:
                pc_final, ci_c_final, r2c_final = pc75, ci_c75, r2c75_or_msg
                cooling_thresholds[hour] = 75
            else:
                return f"[fail] cooling two threshold failed: {county_code}, hour={hour}"

            cooling_stats[hour, :] = np.concatenate([pc_final, ci_c_final, [r2c_final]])

        np.savez(save_path,
            heating=heating_stats,
            cooling=cooling_stats,
            heating_threshold=heating_thresholds,
            cooling_threshold=cooling_thresholds
        )
        return None

    except Exception as e:
        return f"[error] {county_code}: {e}"


def main():
    load_dir = 'your_directory/loads/new'
    weather_dir = 'your_directory/loads/weather'
    output_dir = 'your_directory/loads/com_fitting'
    os.makedirs(output_dir, exist_ok=True)

    tasks = []
    for load_file in sorted(glob.glob(os.path.join(load_dir, 'load_*.npz'))):
        county_code = os.path.splitext(os.path.basename(load_file))[0].split('_')[-1]
        weather_file = os.path.join(weather_dir, f'weather_{county_code}.npz')
        save_file = os.path.join(output_dir, f'fitting_{county_code}.npz')
        if os.path.exists(weather_file):
            tasks.append((load_file, weather_file, save_file))

    print(f"process {len(tasks)} county")
    with ProcessPoolExecutor() as executor:
        for result in tqdm(executor.map(process_one_file, tasks), total=len(tasks)):
            if result:
                print(result)

if __name__ == '__main__':
    main()

In [ ]:
import matplotlib.pyplot as plt
import os

fitting_dir = 'your_directory/loads/com_fitting'
load_dir = 'your_directory/loads/new'
weather_dir = 'your_directory/loads/weather'

fitting_files = sorted([f for f in os.listdir(fitting_dir) if f.endswith('.npz')])
chosen_file = 'fitting_06007.npz'
county_code = chosen_file.replace('fitting_', '').replace('.npz', '')

fitting_data = np.load(os.path.join(fitting_dir, chosen_file))
heating_stats = fitting_data['heating']
cooling_stats = fitting_data['cooling']
heating_thresholds = fitting_data['heating_threshold']
cooling_thresholds = fitting_data['cooling_threshold']

load_data = np.load(os.path.join(load_dir, f'load_{county_code}.npz'))
weather_data = np.load(os.path.join(weather_dir, f'weather_{county_code}.npz'))['data']
temperature = weather_data[0:0 + 8760, 5]

heating_load = load_data['com_heating'] / 1000
cooling_load = load_data['com_cooling'] / 1000

heat_by_hour = heating_load.reshape((365, 24))
cool_by_hour = cooling_load.reshape((365, 24))
temp_by_hour = temperature.reshape((365, 24))

fig, axes = plt.subplots(6, 4, figsize=(18, 20))
axes = axes.flatten()

for hour in range(24):
    ax = axes[hour]

    x = temp_by_hour[:, hour]
    y_heat = heat_by_hour[:, hour]
    y_cool = cool_by_hour[:, hour]

    ax.scatter(x, y_heat, color='red', alpha=0.25, label='Heating')
    ax.scatter(x, y_cool, color='blue', alpha=0.25, label='Cooling')

    ph = heating_stats[hour, :3]
    threshold_h = heating_thresholds[hour]

    if not np.any(np.isnan(ph)) and not np.isnan(threshold_h):
        xh_range = np.linspace(x.min(), threshold_h, 100)
        yh_pred = ph[0]*xh_range**2 + ph[1]*xh_range + ph[2]
        ax.plot(xh_range, yh_pred, color='yellow', lw=2, label=f'Heating Fit (thr={int(threshold_h)})')

    pc = cooling_stats[hour, :3]
    threshold_c = cooling_thresholds[hour]

    if not np.any(np.isnan(pc)) and not np.isnan(threshold_c):
        xc_range = np.linspace(threshold_c, x.max(), 100)
        yc_pred = pc[0]*xc_range**2 + pc[1]*xc_range + pc[2]
        ax.plot(xc_range, yc_pred, color='green', lw=2, label=f'Cooling Fit (thr={int(threshold_c)})')

    ax.set_title(f'Hour {hour:02d}:00')
    ax.set_xlabel('Temp (×10 °C)')
    ax.set_ylabel('Load (MW)')
    ax.grid(True)

fig.suptitle(f'County {county_code} — 24 Hour Load-Temperature Fit', fontsize=20)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
